# Dataset time series viewer

Explore `data/raw/dataset.csv` using a series selector and start/end date pickers.
Run all cells from the project root or any directory inside the repository, using
its Python environment (`uv sync`). Dates use **UTC**, and the end date is inclusive.
The initial view covers the last seven calendar days available in the dataset.
Missing values remain gaps; values are not resampled or filled.

In [ ]:
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "docs/initial_assignment.md").exists()
)
DATA_PATH = ROOT / "data/raw/dataset.csv"
TIME_COLUMN = "validfrom (UTC)"

In [ ]:
df = pd.read_csv(DATA_PATH)
df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], utc=True)
df = df.set_index(TIME_COLUMN).sort_index()
series_columns = df.select_dtypes(include="number").columns.tolist()
first_date = df.index.min().date()
last_date = df.index.max().date()

print(f"Loaded {len(df):,} observations: {first_date} to {last_date} (UTC).")

## Select a series and date range

The plot updates whenever a selection changes. Each series retains the units shown
in its column name. Clear or reversed dates display a message until corrected.

In [ ]:
start_date = widgets.DatePicker(
    description="Start date:",
    value=max(first_date, last_date - pd.Timedelta(days=6)),
    min=first_date,
    max=last_date,
)
end_date = widgets.DatePicker(
    description="End date:", value=last_date, min=first_date, max=last_date,
)
series = widgets.Dropdown(
    description="Series:",
    options=series_columns,
    value="emissionfactor (kg CO2/kWh)",
    layout=widgets.Layout(width="500px"),
)


def plot_series(start, end, column):
    if start is None or end is None:
        print("Choose both a start date and an end date.")
        return
    if start > end:
        print("Start date must be on or before end date.")
        return

    lower = pd.Timestamp(start, tz="UTC")
    upper = pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)
    selected = df.loc[(df.index >= lower) & (df.index < upper), column]
    if selected.empty or selected.isna().all():
        print("No data available for this series and date range.")
        return

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(selected.index, selected, linewidth=1)
    ax.set(title=column, xlabel="Time (UTC)", ylabel=column)
    ax.grid(True, alpha=0.25)
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()
    plt.close(fig)


output = widgets.interactive_output(
    plot_series, {"start": start_date, "end": end_date, "column": series},
)
display(widgets.VBox([widgets.HBox([start_date, end_date]), series, output]))

## Processed dataset

Explore `data/processed/dataset_v1.csv` with the controls below. Blue, orange,
and green show `train`, `valid`, and `test`. Black dots mark replaced values
for the selected source, using its own imputation flag. The target has no
imputed values. The initial view covers the full dataset; dates use UTC.


In [ ]:
processed = pd.read_csv(ROOT / "data/processed/dataset_v1.csv")
processed[TIME_COLUMN] = pd.to_datetime(processed[TIME_COLUMN], utc=True)
processed = processed.set_index(TIME_COLUMN).sort_index()
processed_columns = processed.select_dtypes(include="number").columns.tolist()
processed_first = processed.index.min().date()
processed_last = processed.index.max().date()

processed_start = widgets.DatePicker(
    description="Start date:", value=processed_first,
    min=processed_first, max=processed_last,
)
processed_end = widgets.DatePicker(
    description="End date:", value=processed_last,
    min=processed_first, max=processed_last,
)
processed_series = widgets.Dropdown(
    description="Series:", options=processed_columns,
    value="emissionfactor (kg CO2/kWh)",
    layout=widgets.Layout(width="500px"),
)


def plot_processed_series(start, end, column):
    if start is None or end is None:
        print("Choose both a start date and an end date.")
        return
    if start > end:
        print("Start date must be on or before end date.")
        return

    lower = pd.Timestamp(start, tz="UTC")
    upper = pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1)
    selected = processed.loc[(processed.index >= lower) & (processed.index < upper)]
    if selected.empty:
        print("No data available for this date range.")
        return

    fig, ax = plt.subplots(figsize=(14, 4))
    for split, color in {"train": "tab:blue", "valid": "tab:orange", "test": "tab:green"}.items():
        segment = selected.loc[selected["split"] == split, column]
        if not segment.empty:
            ax.plot(segment.index, segment, color=color, linewidth=1, label=split)

    flag = column.replace("_volume (kWh)", "_is_imputed")
    if flag in selected.columns and flag.endswith("_is_imputed"):
        imputed = selected.loc[selected[flag], column]
        if not imputed.empty:
            ax.scatter(
                imputed.index, imputed, color="black", s=25,
                label="Imputed", zorder=3,
            )
    ax.set(title=column, xlabel="Time (UTC)", ylabel=column)
    ax.legend()
    ax.grid(True, alpha=0.25)
    fig.autofmt_xdate()
    fig.tight_layout()
    plt.show()
    plt.close(fig)


processed_output = widgets.interactive_output(
    plot_processed_series,
    {"start": processed_start, "end": processed_end, "column": processed_series},
)
display(widgets.VBox([
    widgets.HBox([processed_start, processed_end]),
    processed_series,
    processed_output,
]))
